# RSNA Knee: report → label generation (forced-choice assertion scoring)

Self-contained Kaggle notebook. It produces **per-finding assertion probabilities**
(A present / B possible / C not mentioned / D absent / E prior-only) for every
training report, using a local open-weight instruction model. Reports are never sent to an external service.

**What was tested locally (RTX 4050, 6 GB, Windows, torch 2.7.1, transformers 5.5.4, bitsandbytes 0.50.1):**
the labeler code in cell 6 (identical to `src/labeling/llm_assertion_labeler.py`), Qwen3-4B-Instruct-2507 in nf4, the
prefix-cache path (equivalent to the uncached path), and a full 4,407-report run.
**Not tested:** the Kaggle-specific path discovery and installation cells, T4/P100 timings, and models other than Qwen3-4B.

Competition-rule assumptions (see `reports/competition_rules_audit.md`): open-weight public models are allowed as
external data. Reports are training-only data. Internet use is only for installing packages and downloading weights in
**this offline-labeling notebook**, never in the submission notebook.

## 1. Configuration (the only cell you normally edit)

In [ ]:
import os
CONFIG = {
    "MODEL_ID": "Qwen/Qwen3-4B-Instruct-2507",   # or a local path, e.g. /kaggle/input/qwen3-4b-instruct-2507
    "QUANT": "nf4",                # "nf4" (tested) | "none" (fp16/bf16, needs ~9 GB VRAM for 4B)
    "MAX_BATCH_TOKENS": 8000,      # raise to ~24000 on a 16 GB T4 with QUANT="none"
    "MAX_BATCH_SIZE": 4,
    "MAX_REPORT_TOKENS": 6000,     # reports longer than this are truncated AND flagged
    "SMOKE_TEST_N": 5,
    "RUN_FULL": True,              # set False to stop after the smoke test
    "OUT_DIR": "/kaggle/working/labels_v3" if os.path.isdir("/kaggle") else "./labels_v3_out",
}
print(CONFIG)

## 2. Environment inspection

In [ ]:
import sys, shutil, platform, subprocess
import torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| platform", platform.platform())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB  bf16={torch.cuda.is_bf16_supported()}")
else:
    print("!! No GPU: a 4B model on CPU would take many hours; enable a GPU accelerator.")
total, used, free = shutil.disk_usage(".")
print(f"disk free: {free/1e9:.1f} GB")
try:
    import psutil; print(f"RAM: {psutil.virtual_memory().total/1e9:.1f} GB")
except Exception:
    pass

## 3. Dependencies (installs only if missing and internet is on)

In [ ]:
import importlib, socket
def has(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False
def internet():
    try:
        socket.create_connection(("pypi.org", 443), timeout=3); return True
    except OSError:
        return False
need = [m for m in ["transformers", "accelerate", "bitsandbytes", "sklearn", "scipy"] if not has(m)]
print("missing:", need)
if need:
    if internet():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + [{"sklearn": "scikit-learn"}.get(m, m) for m in need], check=False)
    else:
        print("!! Internet is off and packages are missing: attach a wheels dataset and pip install from it.")
import transformers; print("transformers", transformers.__version__)

## 4. Input discovery and data-integrity checks

In [ ]:
import glob
import numpy as np, pandas as pd
cands = glob.glob("/kaggle/input/**/train.csv", recursive=True) + glob.glob("../data/Raw_dicom/train.csv") + glob.glob("data/Raw_dicom/train.csv")
cands = [c for c in cands if "Report" in open(c, encoding="utf-8").readline()]
assert cands, "train.csv with a Report column not found -- attach the competition data"
TRAIN_CSV = cands[0]; print("using", TRAIN_CSV)
df = pd.read_csv(TRAIN_CSV)
TARGETS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
print("studies", len(df), "| unique", df.StudyInstanceUID.nunique())
print("empty reports", int(df.Report.fillna("").str.strip().eq("").sum()))
gold_mask = df[TARGETS].notna().all(axis=1)
print("gold studies", int(gold_mask.sum()), "| unlabeled", int((~gold_mask).sum()))
assert df.StudyInstanceUID.is_unique

## 5. Shared metrics (identical to `src/labeling/metrics.py`)

In [ ]:
"""Evaluation utilities shared by the label audit, the new label pipeline and the
downstream experiments.

All functions take plain numpy arrays. ROC-AUC is reported as NaN (never imputed)
when the evaluation subset contains a single class. Confidence intervals come
from a study-level bootstrap; for macro metrics the same resampled studies are
used for every class, so the interval reflects the joint uncertainty.
"""
import numpy as np
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]


def roc_auc_score(y, s):
    """Mann-Whitney AUC with average ranks for ties (same value as sklearn)."""
    y = np.asarray(y)
    r = rankdata(s)
    n_pos = int((y == 1).sum())
    n_neg = len(y) - n_pos
    return (r[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def safe_auc(y, s):
    y = np.asarray(y, dtype=float)
    s = np.asarray(s, dtype=float)
    m = ~(np.isnan(y) | np.isnan(s))
    y, s = y[m], s[m]
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def ece(y, p, n_bins=5):
    """Expected calibration error with equal-width bins (few bins: n is small)."""
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(p, dtype=float), 0, 1)
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, bins[1:-1]), 0, n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            total += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(total)


def binary_report(y, s, threshold=0.5, n_boot=2000, seed=0):
    """Metrics for one target. `s` may be a probability or any monotone score;
    calibration metrics are only meaningful when it is a probability in [0,1]."""
    y = np.asarray(y, dtype=float)
    s = np.asarray(s, dtype=float)
    m = ~(np.isnan(y) | np.isnan(s))
    y, s = y[m], s[m]
    n = len(y)
    pos = int(y.sum())
    pred = (s >= threshold).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    f1 = 2 * prec * rec / (prec + rec) if (tp and not np.isnan(prec) and not np.isnan(rec)) else (0.0 if tp + fp + fn else np.nan)
    auc = safe_auc(y, s)
    lo = hi = np.nan
    if not np.isnan(auc) and n_boot:
        rng = np.random.default_rng(seed)
        pos_i, neg_i = np.where(y == 1)[0], np.where(y == 0)[0]
        boots = []
        for _ in range(n_boot):  # stratified: keep class counts fixed
            bi = np.concatenate([rng.choice(pos_i, len(pos_i)), rng.choice(neg_i, len(neg_i))])
            boots.append(roc_auc_score(y[bi], s[bi]))
        lo, hi = np.percentile(boots, [2.5, 97.5])
    is_prob = bool(s.min() >= 0 and s.max() <= 1)
    pc = np.clip(s, 1e-4, 1 - 1e-4)
    return {
        "n": n, "n_pos": pos, "n_neg": n - pos,
        "auc": auc, "auc_lo": lo, "auc_hi": hi,
        "ap": float(average_precision_score(y, s)) if 0 < pos < n else np.nan,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "precision": prec, "recall": rec, "specificity": spec, "f1": f1,
        "balanced_acc": np.nanmean([rec, spec]) if not (np.isnan(rec) and np.isnan(spec)) else np.nan,
        "accuracy": (tp + tn) / n if n else np.nan,
        "brier": float(np.mean((s - y) ** 2)) if is_prob else np.nan,
        "log_loss": float(-np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc))) if is_prob else np.nan,
        "ece5": ece(y, s) if is_prob else np.nan,
        "mean_score_pos": float(s[y == 1].mean()) if pos else np.nan,
        "mean_score_neg": float(s[y == 0].mean()) if n - pos else np.nan,
    }


def macro_auc_ci(Y, S, n_boot=2000, seed=0):
    """Macro AUC over columns with both classes, with a joint study bootstrap CI.

    Y, S: (n_studies, n_targets) arrays; NaN in S marks a missing score and is
    excluded per column.
    """
    Y = np.asarray(Y, dtype=float)
    S = np.asarray(S, dtype=float)
    point = np.nanmean([safe_auc(Y[:, j], S[:, j]) for j in range(Y.shape[1])])
    rng = np.random.default_rng(seed)
    n = Y.shape[0]
    boots = []
    for _ in range(n_boot):
        bi = rng.integers(0, n, n)
        vals = [safe_auc(Y[bi, j], S[bi, j]) for j in range(Y.shape[1])]
        vals = [v for v in vals if not np.isnan(v)]
        if vals:
            boots.append(np.mean(vals))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(point), float(lo), float(hi)


def paired_macro_auc_diff(Y, S1, S2, n_boot=2000, seed=0):
    """Bootstrap distribution of macro-AUC(S1) - macro-AUC(S2) on the same studies.
    Returns (point difference, 2.5%, 97.5%, fraction of bootstrap draws <= 0)."""
    Y = np.asarray(Y, dtype=float)
    def mac(S, idx):
        v = [safe_auc(Y[idx, j], S[idx, j]) for j in range(Y.shape[1])]
        v = [x for x in v if not np.isnan(x)]
        return np.mean(v) if v else np.nan
    n = Y.shape[0]
    full = np.arange(n)
    point = mac(S1, full) - mac(S2, full)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        bi = rng.integers(0, n, n)
        d = mac(S1, bi) - mac(S2, bi)
        if not np.isnan(d):
            diffs.append(d)
    diffs = np.array(diffs)
    return float(point), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5)), float((diffs <= 0).mean())

## 6. Labeler (identical to `src/labeling/llm_assertion_labeler.py`, minus its CLI)

In [ ]:
"""Report -> per-finding assertion probabilities with a local instruction LLM.

Method (v3, "forced-choice assertion scoring"):
  * The full report (no character truncation) is placed in a chat prompt that
    defines the 12 competition findings and five assertion categories:
        A PRESENT | B POSSIBLE | C NOT MENTIONED | D ABSENT | E PRIOR ONLY
  * The assistant turn is teacher-forced to the fixed layout
        "ACL injury: X\nMCL injury: X\n..."
    and at every X the next-token distribution restricted to the five letter
    tokens is recorded. The argmax letter is fed back before the next item, so
    the model sees its own previous answers exactly as in greedy decoding.
  * Nothing is parsed from free text, so there is no JSON failure mode and no
    0.5 fallback. `letter_mass` (probability mass the model put on the five
    letters before renormalisation) is stored as a format-adherence check; rows
    below `--min-letter-mass` are flagged, not silently accepted.

Outputs (append-only CSV, resumable, one row per study):
  StudyInstanceUID, status, error, n_prompt_tokens, report_truncated,
  {target}__pA..pE, {target}__letter_mass, {target}__argmax, run metadata.

The reports never leave this machine.
"""
import argparse
import hashlib
import json
import os
import time
import traceback
import unicodedata

import numpy as np
import pandas as pd
import torch
from transformers import DynamicCache

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

# Item labels used in the forced assistant layout (order == TARGETS).
ITEM_KEYS = ["ACL injury", "MCL injury", "Medial meniscus tear", "Lateral meniscus tear",
             "Medial tibiofemoral OA", "Lateral tibiofemoral OA", "Patellofemoral OA",
             "Joint effusion", "Synovitis", "Baker's cyst", "Bone contusion", "Fracture"]
LETTERS = ["A", "B", "C", "D", "E"]
PROMPT_VERSION = "assert-v3.1"

# 95% of the Greek reports spell mu with the MICRO SIGN and Delta with the
# INCREMENT sign (PDF/Word conversion artefact); map them back to Greek letters.
GREEK_FIXES = {"µ": "μ", "∆": "Δ", "Ω": "Ω"}


def normalize_report(text):
    text = unicodedata.normalize("NFC", str(text))
    n_greek = sum(1 for c in text if "Ͱ" <= c <= "Ͽ" or "ἀ" <= c <= "῿")
    n_latin = sum(1 for c in text if c.isascii() and c.isalpha())
    if n_greek > 0.3 * max(n_latin, 1):
        for a, b in GREEK_FIXES.items():
            text = text.replace(a, b)
    return text

SYSTEM_PROMPT = """You are a meticulous musculoskeletal radiologist. You read knee MRI reports written in any language (English, Spanish, Turkish, Croatian/Bosnian/Serbian, Greek, German, Bulgarian, Dutch, French, and others) and state, for one finding at a time, what the report says about it.

Answer every item with exactly one letter:
A = PRESENT: the report states the finding is currently present (any grade or size, including partial, small, mild, or chronic but still present).
B = POSSIBLE: the report calls it possible, suspected, probable, equivocal or "cannot be excluded", or describes changes that only partly meet the definition below.
C = NOT MENTIONED: nothing in the report addresses this finding or the structure it concerns.
D = ABSENT: the report states it is absent, normal or intact, or describes the relevant structure as normal.
E = PRIOR ONLY: the finding is only historical (operated, repaired, reconstructed, healed or resolved) and is not currently present.

Reading rules:
- Use the findings and the impression/conclusion. Clinical history, indication or the clinical question are NOT evidence: "rule out meniscal tear", "trauma", "contusion of the knee" or "suspected ACL tear?" written as the reason for the exam does not make a finding present.
- If findings and impression disagree, prefer the impression/conclusion.
- A statement that a structure is normal/intact counts as D for findings of that structure.
- Inside the findings or impression, "R/O X", "rule out X", "DD X", "to exclude X" or "X?" means X is suspected (B), not excluded.
- Each report describes one knee. "Medial" and "lateral" refer to the compartments of that knee.

Finding definitions:
1. ACL injury: current tear of the anterior cruciate ligament (complete, partial, high- or low-grade), rupture, sprain, avulsion, or a torn ACL graft. Mucoid/myxoid degeneration, ganglion cyst or thickening without a tear = B at most. Intact graft after reconstruction = E.
2. MCL injury: sprain of any grade (including grade 1 periligamentous edema), partial or complete tear, or avulsion of the medial collateral ligament. Old post-traumatic thickening/ossification only (e.g. Pellegrini-Stieda) = E.
3. Medial meniscus tear: any tear of the medial meniscus (horizontal, vertical, longitudinal, radial, oblique, complex, flap, bucket-handle, root tear, meniscocapsular separation), grade 3 signal reaching the articular surface (also when called "grade III degeneration" or "grade III meniscopathy"), maceration or fragmentation. Grade 1-2 intrasubstance signal/degeneration, extrusion alone or a discoid meniscus without tear = D. Partial meniscectomy without a new tear = E.
4. Lateral meniscus tear: same rules for the lateral meniscus.
5. Medial tibiofemoral OA: osteoarthritis (arthrosis, gonarthrosis, degenerative arthropathy) of the medial femorotibial compartment, or advanced cartilage loss there (grade 3-4 chondromalacia/chondropathy, full-thickness defects, denuded bone), especially with osteophytes, subchondral sclerosis/cysts/edema or joint-space narrowing. Only mild (grade 1-2) chondromalacia = B. Generalized/tricompartmental/femorotibial OA counts for both medial and lateral compartments.
6. Lateral tibiofemoral OA: same rules for the lateral femorotibial compartment.
7. Patellofemoral OA: osteoarthritis of the patellofemoral joint, or advanced cartilage loss of the patella or trochlea (grade 3-4, full-thickness ulcers), especially with osteophytes or subchondral changes. Only mild (grade 1-2) chondromalacia patellae = B.
8. Joint effusion: increased intra-articular fluid of any amount (small, mild, moderate, large), hemarthrosis or lipohemarthrosis. Fluid described as physiological or normal = D. Fluid only in a Baker's cyst or a bursa does not count.
9. Synovitis: inflammation of the joint lining: synovitis, synovial thickening, proliferation or hypertrophy, effusion-synovitis, Hoffa synovitis, PVNS. Hoffa fat-pad edema or impingement without the words synovitis/synovial thickening = B. Plica alone = D unless thickened/inflamed.
10. Baker's cyst: popliteal (Baker's) cyst / distended gastrocnemius-semimembranosus bursa, including ruptured or leaking cysts. Ganglion cysts, parameniscal cysts and other bursae do not count.
11. Bone contusion: bone bruise / contusion / traumatic bone-marrow edema. Bone-marrow edema that is clearly non-traumatic (degenerative subchondral edema, insufficiency) = B. Red-marrow reconversion/hyperplasia, enchondroma, bone cysts or a normal marrow signal are not contusion (D).
12. Fracture: any current fracture of the knee bones, including osteochondral, impaction, avulsion (e.g. Segond), stress or subchondral insufficiency fracture. Old healed fracture = E. Microtrabecular injury described only as bone bruise = B.

Common terms: tear = rotura/desgarro (es), yırtık (tr), ruptura/puknuće (hr), ρήξη (el), Riss/Ruptur (de), руптура/разкъсване (bg), scheur (nl), rupture/fissure/déchirure (fr). Effusion = derrame (es), sıvı artışı/efüzyon (tr), izljev (hr), συλλογή υγρού (el), Erguss (de), излив (bg), hydrops/vocht (nl), épanchement (fr). Bone marrow edema/bruise = edema óseo/contusión ósea, kemik iliği ödemi/kontüzyon, edem kosti/kontuzija, οίδημα μυελού/οστική θλάση, Knochenmarködem/bone bruise, костномозъчен едем/контузия, botoedeem/bone bruise, œdème osseux/contusion osseuse."""

USER_TEMPLATE = """Knee MRI report:
<<<
{report}
>>>

Answer all 12 items with one letter each (A, B, C, D or E), in this exact layout:
{layout}"""


def build_layout():
    return "\n".join(f"{k}: <letter>" for k in ITEM_KEYS)


def load_model(model_id, quant, cache_dir):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
    kw = {"dtype": torch.bfloat16, "device_map": "cuda:0", "cache_dir": cache_dir}
    if quant == "nf4":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    elif quant == "int8":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    model.eval()
    return tok, model


def single_token_id(tok, text):
    ids = tok.encode(text, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"{text!r} is not a single token: {ids}")
    return ids[0]


class Scorer:
    """Scores reports. With `prefix_cache=True` the constant part of the prompt
    (system message + start of the user turn) is encoded once; each batch only
    prefills the report-specific tokens, padded between the cached prefix and
    the report ("middle padding") with explicit RoPE position ids."""

    SENTINEL = "⁣@@REPORT@@⁣"

    def __init__(self, tok, model, max_report_tokens=6000, prefix_cache=True):
        self.tok, self.model = tok, model
        self.max_report_tokens = max_report_tokens
        self.letter_ids = [single_token_id(tok, " " + L) for L in LETTERS]
        # forced segments: first item directly after the assistant header, later
        # items after a newline that follows the previous answer letter.
        self.first_ids = tok.encode(ITEM_KEYS[0] + ":", add_special_tokens=False)
        self.next_ids = [tok.encode("\n" + k + ":", add_special_tokens=False) for k in ITEM_KEYS[1:]]
        self.pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
        msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_TEMPLATE.format(report=self.SENTINEL, layout=build_layout())}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        pre, post = text.split(self.SENTINEL)
        self.prefix_ids = tok.encode(pre, add_special_tokens=False)
        self.post_ids = tok.encode(post, add_special_tokens=False) + self.first_ids
        self.prefix_cache = prefix_cache
        self.prefix_kv = None
        if prefix_cache:
            with torch.no_grad():
                out = model(input_ids=torch.tensor([self.prefix_ids], device=model.device),
                            use_cache=True, logits_to_keep=1)
            self.prefix_kv = [(lay.keys, lay.values) for lay in out.past_key_values.layers]

    def prompt_ids(self, report):
        """Report-specific token ids (everything after the shared prefix)."""
        report = normalize_report(report)
        rep_ids = self.tok.encode(report, add_special_tokens=False)
        truncated = len(rep_ids) > self.max_report_tokens
        if truncated:  # never silent: flagged in the output
            rep_ids = rep_ids[: self.max_report_tokens]
        return rep_ids + self.post_ids, truncated

    @torch.no_grad()
    def score_batch(self, prompts):
        """prompts: list of report-specific token-id lists (see prompt_ids).
        Returns (B, 12, 5) renormalised letter probabilities and (B, 12) letter mass."""
        dev = self.model.device
        B = len(prompts)
        if not self.prefix_cache:  # full prompts, plain left padding
            prompts = [self.prefix_ids + p for p in prompts]
        L = max(len(p) for p in prompts)
        ids = torch.full((B, L), self.pad_id, dtype=torch.long)
        att = torch.zeros((B, L), dtype=torch.long)
        for i, p in enumerate(prompts):  # left padding (after the cached prefix, if any)
            ids[i, L - len(p):] = torch.tensor(p)
            att[i, L - len(p):] = 1
        ids, att = ids.to(dev), att.to(dev)
        pos = (att.cumsum(-1) - 1).clamp(min=0)
        past = None
        if self.prefix_cache:
            P = len(self.prefix_ids)
            past = DynamicCache(config=self.model.config)
            for li, (k, v) in enumerate(self.prefix_kv):
                past.update(k.expand(B, -1, -1, -1).contiguous(), v.expand(B, -1, -1, -1).contiguous(), li)
            att = torch.cat([torch.ones((B, P), dtype=att.dtype, device=dev), att], dim=1)
            pos = pos + P
        out = self.model(input_ids=ids, attention_mask=att, position_ids=pos,
                         past_key_values=past, use_cache=True, logits_to_keep=1)
        past = out.past_key_values
        logits = out.logits[:, -1, :].float()
        probs_all = np.zeros((B, len(ITEM_KEYS), len(LETTERS)), dtype=np.float32)
        mass_all = np.zeros((B, len(ITEM_KEYS)), dtype=np.float32)
        next_pos = pos[:, -1] + 1
        for k in range(len(ITEM_KEYS)):
            p_full = torch.softmax(logits, dim=-1)
            p_let = p_full[:, self.letter_ids]
            mass = p_let.sum(-1)
            p_norm = p_let / mass.clamp(min=1e-12).unsqueeze(-1)
            probs_all[:, k, :] = p_norm.cpu().numpy()
            mass_all[:, k] = mass.cpu().numpy()
            if k == len(ITEM_KEYS) - 1:
                break
            choice = torch.tensor(self.letter_ids, device=dev)[p_norm.argmax(-1)]
            seg = torch.tensor(self.next_ids[k], device=dev).unsqueeze(0).expand(B, -1)
            step_ids = torch.cat([choice.unsqueeze(1), seg], dim=1)
            n_new = step_ids.shape[1]
            att = torch.cat([att, torch.ones((B, n_new), dtype=att.dtype, device=dev)], dim=1)
            step_pos = next_pos.unsqueeze(1) + torch.arange(n_new, device=dev).unsqueeze(0)
            out = self.model(input_ids=step_ids, attention_mask=att, position_ids=step_pos,
                             past_key_values=past, use_cache=True, logits_to_keep=1)
            past = out.past_key_values
            logits = out.logits[:, -1, :].float()
            next_pos = next_pos + n_new
        del past, out
        return probs_all, mass_all


def make_batches(lengths, max_batch_tokens, max_batch_size):
    order = np.argsort(lengths)[::-1]  # longest first: early OOM if any
    batches, cur, cur_max = [], [], 0
    for i in order:
        new_max = max(cur_max, lengths[i])
        if cur and (new_max * (len(cur) + 1) > max_batch_tokens or len(cur) >= max_batch_size):
            batches.append(cur)
            cur, cur_max = [], 0
            new_max = lengths[i]
        cur.append(int(i))
        cur_max = new_max
    if cur:
        batches.append(cur)
    return batches

## 7. Load the model

In [ ]:
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)
tok, model = load_model(CONFIG["MODEL_ID"], CONFIG["QUANT"], cache_dir=None)
scorer = Scorer(tok, model, CONFIG["MAX_REPORT_TOKENS"], prefix_cache=True)
print("prefix tokens:", len(scorer.prefix_ids), "| VRAM after load: %.2f GB" % (torch.cuda.memory_allocated()/1e9))

## 8. Smoke test (non-gold reports only). Letter mass must be close to 1.

In [ ]:
smoke = df[~gold_mask].head(CONFIG["SMOKE_TEST_N"])
enc = [scorer.prompt_ids(r)[0] for r in smoke.Report]
probs, mass = scorer.score_batch(enc)
print("letter mass min:", float(mass.min()))
assert mass.min() > 0.5, "model is not following the forced answer format -- check the chat template / model"
for i, uid in enumerate(smoke.StudyInstanceUID):
    print(uid[-12:], " ".join(f"{t[:6]}:{LETTERS[probs[i,k].argmax()]}" for k, t in enumerate(TARGETS)))

## 9. Full run with checkpointing / resume (append-only CSV; re-running skips finished studies)

In [ ]:
import time
OUT_CSV = os.path.join(CONFIG["OUT_DIR"], "llm_assert_raw.csv")
done = set()
if os.path.exists(OUT_CSV):
    prev = pd.read_csv(OUT_CSV, usecols=["StudyInstanceUID", "status"])
    done = set(prev.loc[prev.status == "ok", "StudyInstanceUID"])
todo = df[~df.StudyInstanceUID.isin(done)].reset_index(drop=True)
print("already ok:", len(done), "| todo:", len(todo))
meta = {"model": CONFIG["MODEL_ID"], "quant": CONFIG["QUANT"], "prompt_version": PROMPT_VERSION}
if CONFIG["RUN_FULL"] and len(todo):
    enc = [scorer.prompt_ids(str(r)) for r in todo.Report]
    lengths = np.array([len(e[0]) + len(scorer.prefix_ids) for e in enc])
    batches = make_batches(lengths, CONFIG["MAX_BATCH_TOKENS"], CONFIG["MAX_BATCH_SIZE"])
    t0 = time.time(); n = 0
    for bi, idx in enumerate(batches):
        rows = []
        try:
            probs, mass = scorer.score_batch([enc[i][0] for i in idx])
            for j, i in enumerate(idx):
                r = {"StudyInstanceUID": todo.StudyInstanceUID[i], "status": "ok", "error": "",
                     "n_prompt_tokens": int(lengths[i]), "report_truncated": bool(enc[i][1]), **meta}
                for k, t in enumerate(TARGETS):
                    for li, L in enumerate(LETTERS):
                        r[f"{t}__p{L}"] = round(float(probs[j, k, li]), 5)
                    r[f"{t}__letter_mass"] = round(float(mass[j, k]), 4)
                    r[f"{t}__argmax"] = LETTERS[int(probs[j, k].argmax())]
                if mass[j].min() < 0.5:
                    r["status"] = "low_letter_mass"
                rows.append(r)
        except Exception as ex:  # never silently drop a study
            torch.cuda.empty_cache()
            for i in idx:
                rows.append({"StudyInstanceUID": todo.StudyInstanceUID[i], "status": "failed",
                             "error": f"{type(ex).__name__}: {str(ex)[:200]}", **meta})
        pd.DataFrame(rows).to_csv(OUT_CSV, mode="a", index=False, header=not os.path.exists(OUT_CSV))
        n += len(idx)
        if bi % 25 == 0:
            el = time.time() - t0
            print(f"{n}/{len(todo)}  {n/el:.2f} studies/s  eta {(len(todo)-n)/(n/el)/60:.0f} min")
    print("finished in %.1f min" % ((time.time() - t0) / 60))

## 10. Output validation (counts, duplicates, ranges, failures)

In [ ]:
out = pd.read_csv(OUT_CSV).drop_duplicates("StudyInstanceUID", keep="last")
print("rows", len(out), "| expected", len(df), "| missing", len(set(df.StudyInstanceUID) - set(out.StudyInstanceUID)))
print(out.status.value_counts().to_dict())
pc = [c for c in out.columns if "__p" in c]
assert out.loc[out.status == "ok", pc].notna().all().all(), "NaN probabilities in ok rows"
assert ((out[pc].fillna(0) >= 0) & (out[pc].fillna(0) <= 1)).all().all()
failed = out[out.status != "ok"][["StudyInstanceUID", "status", "error"]]
failed.to_csv(os.path.join(CONFIG["OUT_DIR"], "failed_or_flagged.csv"), index=False)
print("failed/flagged:", len(failed), "| truncated reports:", int(out.report_truncated.fillna(False).sum()))

## 11. Gold validation (information only: never tune prompts or thresholds on these numbers)
Score used: `s0 = pA + 0.5*pB + 0.25*pE` (fixed a priori).

In [ ]:
g = df[gold_mask].set_index("StudyInstanceUID")[TARGETS].astype(float)
o = out.set_index("StudyInstanceUID").reindex(g.index)
S = pd.DataFrame({t: o[f"{t}__pA"] + 0.5 * o[f"{t}__pB"] + 0.25 * o[f"{t}__pE"] for t in TARGETS})
m = S.notna().all(axis=1)
mp, lo, hi = macro_auc_ci(g[m].values, S[m].values, n_boot=2000)
print(f"gold macro AUC {mp:.3f} [{lo:.3f}, {hi:.3f}] on n={int(m.sum())}")
for t in TARGETS:
    r = binary_report(g.loc[m, t].values, S.loc[m, t].values, n_boot=0)
    print(f"{t:18s} AUC {r['auc']:.3f}  sens {r['recall']:.2f}  spec {r['specificity']:.2f}  (pos {r['n_pos']})")

## 12. Export

In [ ]:
import zipfile
zp = os.path.join(CONFIG["OUT_DIR"], "labels_v3_export.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(CONFIG["OUT_DIR"]):
        if f.endswith(".csv"):
            z.write(os.path.join(CONFIG["OUT_DIR"], f), f)
print("saved", zp, "-- download it from the Output tab (or commit the notebook to persist /kaggle/working).")

## 13. Runtime / memory notes and troubleshooting
* Measured locally: about 0.55–0.6 studies/s on an RTX 4050 6 GB (nf4, batch ≈ 4). A T4 16 GB with `QUANT="none"`
  and `MAX_BATCH_TOKENS≈24000` should be faster; that estimate is untested.
* `letter mass` near 0 means the model does not follow the forced layout (wrong chat template or a thinking model).
  Use an *instruct* (non-thinking) model, or disable thinking in the template.
* OOM: lower `MAX_BATCH_TOKENS`. The code retries nothing silently: failed rows are written with `status=failed`;
  re-running the cell retries only those.
* Never feed any report-derived value into the image-model submission notebook. Test studies have no report.